# 05 - Normalização para schema canônico

Recebe `configurations_raw.parquet` (17 032 linhas, uma por métrica extraída) e produz
`experiments.parquet` (uma linha por experimento, com todas as métricas como colunas separadas).

Pipeline neste notebook:
1. Carregamento e merge com ano do paper
2. Parsing numérico (`dataset_size`, `num_classes`, `imbalance_ratio`)
3. Normalização por similaridade:  
3.1. Salvar as raw keys únicas para LLM externa  
3.2. LLM auxiliar: mapeia as raw keys para canônico  
3.3. Normalização dos task_types  
4. Aplicação dos mapeamentos LLM ao DataFrame
5. Normalização de métricas e `metric_split`
6. Pivot de métricas → uma linha por experimento
7. Colunas derivadas, critérios de aceitação e output final

In [37]:
import os
import json
import re
from pathlib import Path
from typing import Optional

import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pydantic import BaseModel, Field, ValidationError

# ── Paths ─────────────────────────────────────────────────────────────────────
NB_DIR        = Path().resolve()
PROJECT_DIR   = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
RAW_DIR       = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

CKPT_SOURCE  = PROCESSED_DIR / "extraction_checkpoints.jsonl"
INPUT_RAW    = PROCESSED_DIR / "configurations_raw.parquet"
INPUT_PAPERS = PROCESSED_DIR / "papers_with_pdf.parquet"

RAW_DATASETS_OUT   = RAW_DIR / "raw_mapped_datasets.jsonl"
RAW_MODELS_OUT     = RAW_DIR / "raw_mapped_models.jsonl"
RAW_STRATEGIES_OUT = RAW_DIR / "raw_mapped_strategies.jsonl"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print("Paths OK")

Paths OK


## 1. Carregamento e merge

In [38]:
df_raw    = pd.read_parquet(INPUT_RAW)
df_papers = pd.read_parquet(INPUT_PAPERS)[["paper_id", "year"]]

df = df_raw.merge(df_papers, on="paper_id", how="left")

print(f"Configurations: {len(df):,}  |  Papers: {df['paper_id'].nunique():,}  |  Year coverage: {df['year'].notna().mean():.1%}")

Configurations: 17,032  |  Papers: 512  |  Year coverage: 100.0%


## 2. Parsing numérico

In [39]:
def parse_dataset_size(s) -> Optional[int]:
    if pd.isna(s) or not s:
        return None
    m = re.search(r"\d+", re.sub(r"[,_]", "", str(s)))
    return int(m.group()) if m else None

def parse_num_classes(s) -> Optional[int]:
    if pd.isna(s) or not s:
        return None
    m = re.search(r"\d+", str(s))
    return int(m.group()) if m else None

def parse_imbalance_ratio(s) -> Optional[float]:
    """Handles: IR=100, 100:1, 1:100, 100.0 — always returns max/min (≥1)."""
    if pd.isna(s) or not s:
        return None
    raw = str(s)
    m = re.search(r"IR\s*[=:]\s*([\d.]+)", raw, re.I)
    if m:
        return float(m.group(1))
    m = re.search(r"([\d.]+)\s*:\s*([\d.]+)", raw)
    if m:
        a, b = float(m.group(1)), float(m.group(2))
        if min(a, b) > 0:
            return max(a, b) / min(a, b)
    m = re.search(r"[\d.]+", raw)
    if m:
        v = float(m.group())
        return v if v >= 1.0 else None
    return None

df["dataset_size"]            = df["dataset_size_raw"].map(parse_dataset_size)
df["dataset_num_classes"]     = df["dataset_num_classes_raw"].map(parse_num_classes)
df["dataset_imbalance_ratio"] = df["dataset_imbalance_ratio_raw"].map(parse_imbalance_ratio)
df["dataset_is_multilabel"]   = df["task_type_raw"].str.lower().str.contains(
    r"multi.?label", na=False, regex=True
)

for col in ["dataset_size", "dataset_num_classes", "dataset_imbalance_ratio"]:
    filled = df[col].notna().sum()
    print(f"  {col:<30} {filled:>6,} / {len(df):,}  ({filled/len(df):.1%})")

  dataset_size                    6,010 / 17,032  (35.3%)
  dataset_num_classes             9,201 / 17,032  (54.0%)
  dataset_imbalance_ratio         7,752 / 17,032  (45.5%)


## 3. Mapeamento por chaves únicas
### 3.1 Salvar as raw keys únicas para LLM externa

In [40]:
unique_datasets_raw: set[str] = set()
with open(CKPT_SOURCE, encoding="utf-8") as _f:
    for _line in _f:
        try:
            _entry = json.loads(_line)
            for _cfg in (_entry.get("configurations") or []):
                _d = _cfg.get("dataset_name_raw")
                if _d:
                    unique_datasets_raw.add(_d)
        except Exception:
            pass
        
datasets_json = {}
for dataset in unique_datasets_raw:
    datasets_json[dataset] = []

with open(RAW_DATASETS_OUT, "w", encoding="utf-8") as _f:
    json.dump(datasets_json, _f, ensure_ascii=False, indent=4, sort_keys=True)

print(f"Saved: {RAW_DATASETS_OUT}")
print(f"Total unique dataset names: {len(unique_datasets_raw)}")

Saved: D:\programas\ufmg\ufmg-2026-1\causal_project\data\raw\raw_mapped_datasets.jsonl
Total unique dataset names: 962


In [41]:
unique_models_raw: set[str] = set()
with open(CKPT_SOURCE, encoding="utf-8") as _f:
    for _line in _f:
        try:
            _entry = json.loads(_line)
            for _cfg in (_entry.get("configurations") or []):
                _m = _cfg.get("model_name_raw")
                if _m:
                    unique_models_raw.add(_m)
        except Exception:
            pass
models_format = {
    "transformer": [],
    "cnn": [],
    "rnn": [],
    "gnn": [],
    "gbm": [],
    "ensemble": [],
    "tree": [],
    "mlp": [],
    "kernel": [],
    "linear": [],
    "other": list(unique_models_raw)
}

with open(RAW_MODELS_OUT, "w", encoding="utf-8") as _f:
    json.dump(models_format, _f, ensure_ascii=False, indent=4)

print(f"Saved: {RAW_MODELS_OUT}")
print(f"Total unique model names: {len(unique_models_raw)}")

Saved: D:\programas\ufmg\ufmg-2026-1\causal_project\data\raw\raw_mapped_models.jsonl
Total unique model names: 842


In [42]:
unique_strategies_raw: set[str] = set()
with open(CKPT_SOURCE, encoding="utf-8") as _f:
    for _line in _f:
        try:
            _entry = json.loads(_line)
            for _cfg in (_entry.get("configurations") or []):
                _s = _cfg.get("balancing_strategy_raw")
                if _s:
                    unique_strategies_raw.add(_s)
        except Exception:
            pass

strategies_format = {
	"none": [],
	"oversampling": [],
	"undersampling": [],
	"hybrid": [],
	"cost_sensitive": [],
	"data_augmentation": [],
	"ensemble_based": [],
	"threshold_moving": [],
	"generative": [],
	"two_stage": [],
	"other": list(unique_strategies_raw),
}

with open(RAW_STRATEGIES_OUT, "w", encoding="utf-8") as _f:
    json.dump(strategies_format, _f, ensure_ascii=False, indent=4)

print(f"Saved: {RAW_STRATEGIES_OUT}")
print(f"Total unique balancing strategies: {len(unique_strategies_raw)}")

Saved: D:\programas\ufmg\ufmg-2026-1\causal_project\data\raw\raw_mapped_strategies.jsonl
Total unique balancing strategies: 1663


## 3.2. Mapeamento por LLM externa
Foram passados os arquivos raw gerados pelas últimas células para um LLM, com o prompt:  
> "Map all entries to a canonical name (or default keys if available), if possible. If not, keep the original name. Output in JSON".

Os resultados foram salvos em `processed/mapped_datasets.jsonl`, `processed/mapped_models.jsonl` e `processed/mapped_strategies.jsonl`, respectivamente.

## 3.3. Normalização de `task_type`

Mapeamento por regras (regex) de `task_type_raw` → enum canônico:
`vision / text / tabular / audio / time_series / medical_imaging / other`

A ordem das regras importa: `medical_imaging` antes de `vision` para evitar que histologia/radiologia caia em `vision`; `time_series` e `audio` antes dos genéricos.

In [43]:
_TASK_RULES = [
    ("time_series",     r"time.?series|forecasting|temporal"),
    ("audio",           r"audio|speech|sound"),
    ("medical_imaging", r"medical.?imag|histolog|patholog|radiolog|dermoscop|fundus|chest.?x.?ray|\bmri\b|\bct\b scan|ecg|eeg|clinical.?imag"),
    ("vision",          r"image|vision|visual|object.?detect|segmentation|long.?tail|land.?cover"),
    ("text",            r"text|sentiment|ner|named.?entity|document|language|\bnlp\b|token.?class|sentence.?class|emotion"),
    ("other",           r"node.?class|graph|link.?pred|ood|out.?of.?distrib|incremental"),
    ("tabular",         r"classif|class\b|\bbinary\b|predict|recogni|detect"),
]

def normalize_task_type(raw) -> str:
    if pd.isna(raw) or not str(raw).strip():
        return "other"
    s = str(raw).lower().strip()
    for label, pattern in _TASK_RULES:
        if re.search(pattern, s):
            return label
    return "other"

df["task_type"] = df["task_type_raw"].map(normalize_task_type)

print(df["task_type"].value_counts().to_string())

task_type
other          8961
tabular        6968
vision          764
text            261
time_series      78


## 4. Aplicação dos mapeamentos LLM ao DataFrame

Carrega os três JSONs de mapeamento e aplica diretamente ao `df`:
- `dataset_name_raw` → `dataset_canonical`
- `model_name_raw` → `model_family` (família) + `model_specific` (nome literal)
- `balancing_strategy_raw` → `balancing_strategy` (categoria) + `balancing_strategy_specific` (literal)

In [44]:
with open(PROCESSED_DIR / "mapped_models.json", encoding="utf-8") as f:
    mapped_models = json.load(f)
model_map = {raw: family for family, raws in mapped_models.items() for raw in raws}

with open(PROCESSED_DIR / "mapped_datasets.json", encoding="utf-8") as f:
    mapped_datasets = json.load(f)
dataset_map = {raw: can for can, raws in mapped_datasets.items() for raw in raws}

with open(PROCESSED_DIR / "mapped_strategies.json", encoding="utf-8") as f:
    mapped_strategies = json.load(f)
strategy_map = {raw: cat for cat, raws in mapped_strategies.items() for raw in raws}

df["model_family"]                = df["model_name_raw"].map(model_map)
df["model_specific"]              = df["model_name_raw"]
df["dataset_canonical"]           = df["dataset_name_raw"].map(dataset_map).fillna(df["dataset_name_raw"])
df["balancing_strategy"]          = df["balancing_strategy_raw"].map(strategy_map)
df["balancing_strategy_specific"] = df["balancing_strategy_raw"]

print(f"model_family coverage: {df['model_family'].notna().mean():.1%}")
print(df["model_family"].value_counts().to_string())
print()
print(df["balancing_strategy"].value_counts().to_string())

model_family coverage: 98.3%
model_family
other          3605
ensemble       3080
cnn            2910
kernel         2035
tree           1869
mlp            1383
gbm             887
linear          665
transformer     128
rnn              93
gnn              88

balancing_strategy
oversampling         4990
other                4561
none                 2929
cost_sensitive       1411
ensemble_based       1074
undersampling         966
threshold_moving      433
hybrid                306
generative            206
data_augmentation     144
two_stage              12


## 5. Normalização de métricas e `metric_split`

Mapeia `metric_name_raw` → coluna canônica e `metric_split_raw` → enum `test / val / cv / train`.
Descarta linhas sem métrica reconhecida ou sem split reconhecido — o restante segue para o pivot.

In [45]:
_METRIC_RULES: list[tuple[str, str]] = [
    ("metric_f1_macro",     r"macro.*(f1|f.measure|f.score)|f1.*macro|\bmf1\b|macro.*averaged.*f"),
    ("metric_f1_weighted",  r"weighted.*(f1|f.score)|f1.*weighted"),
    ("metric_f1_micro",     r"micro.*(f1|f.score)|f1.*micro"),
    ("metric_f1_binary",    r"^f1[\-. ]?score$|^f1[\-. ]?measure$|^f1$|^f[\-. ]?score$|^f[\-. ]?measure$|^fm$|^f[\-. ]?score averaged"),
    ("metric_balanced_acc", r"balanced.acc|balanc.*error|\bbacc\b|\bbtest acc"),
    ("metric_accuracy",     r"^acc$|^accuracy$|top.?1.acc|overall.acc|test.acc|average.acc|\bavacc\b|\bacsa\b"),
    ("metric_aucroc",       r"auc.roc|roc.auc|\bauroc\b|^roc$|^auc$|roc.area|\bmauc\b|^auroc\b"),
    ("metric_auprc",        r"\bauprc\b|\baucprc\b|\baupr\b|pr.auc|auc.pr|\bap\b(?!.*acc)"),
    ("metric_gmean",        r"g.mean|gmean|g.measure|geometric.mean|\bgm\b|\bgmeans\b"),
    ("metric_mcc",          r"^mcc$|matthews"),
    ("metric_tpr_gap",      r"tpr.gap|tpr.diff"),
]

_SPLIT_RULES: list[tuple[str, str]] = [
    ("cv",    r"cross.val|fold|loocv|mccv|bootstrap|\bcv\b"),
    ("train", r"^train"),
    ("val",   r"^val|^dev|internal.val"),
    ("test",  r"test|holdout|hold.out|split|standard|random.split|\d+:\d+"),
]

def normalize_metric_name(raw) -> Optional[str]:
    if pd.isna(raw) or not str(raw).strip():
        return None
    s = str(raw).lower().strip()
    for col, pattern in _METRIC_RULES:
        if re.search(pattern, s):
            return col
    return None

def normalize_split(raw) -> Optional[str]:
    if pd.isna(raw) or not str(raw).strip():
        return None
    s = str(raw).lower().strip()
    for label, pattern in _SPLIT_RULES:
        if re.search(pattern, s):
            return label
    return None

df["metric_col"]   = df["metric_name_raw"].map(normalize_metric_name)
df["metric_split"] = df["metric_split_raw"].map(normalize_split)

df_metrics = df[df["metric_col"].notna() & df["metric_split"].notna()].copy()

print(f"Rows kept (metric + split): {len(df_metrics):,} / {len(df):,}")
print(df_metrics["metric_col"].value_counts().to_string())

Rows kept (metric + split): 4,919 / 17,032
metric_col
metric_accuracy        1768
metric_aucroc           997
metric_f1_binary        840
metric_gmean            536
metric_balanced_acc     207
metric_auprc            185
metric_f1_macro         178
metric_mcc              130
metric_f1_micro          70
metric_f1_weighted        8


## 6. Pivot de métricas → uma linha por experimento

Agrupa por `(paper_id, dataset_canonical, model_family, balancing_strategy, is_baseline_raw, metric_split)` e pivota as métricas em colunas separadas. Quando há duplicatas (mesma métrica no mesmo grupo), toma a média.

In [46]:
METRIC_COLS = [
    "metric_f1_macro", "metric_f1_weighted", "metric_f1_binary", "metric_f1_micro",
    "metric_balanced_acc", "metric_accuracy", "metric_aucroc", "metric_auprc",
    "metric_gmean", "metric_mcc", "metric_tpr_gap",
]

EXP_KEY = ["paper_id", "dataset_canonical", "model_family", "balancing_strategy",
           "is_baseline_raw", "metric_split"]

pivoted = (
    df_metrics
    .pivot_table(index=EXP_KEY, columns="metric_col", values="metric_value", aggfunc="mean")
    .reset_index()
)
for col in METRIC_COLS:
    if col not in pivoted.columns:
        pivoted[col] = np.nan

meta_cols = EXP_KEY + [
    "year", "task_type", "dataset_size", "dataset_num_classes",
    "dataset_imbalance_ratio", "dataset_is_multilabel",
    "model_specific", "balancing_strategy_specific",
]
meta = df_metrics[meta_cols].groupby(EXP_KEY, as_index=False).first()
exp  = meta.merge(pivoted, on=EXP_KEY, how="left")

print(f"Experiments after pivot: {len(exp):,}  |  Papers: {exp['paper_id'].nunique():,}")
print(exp["metric_split"].value_counts().to_string())

Experiments after pivot: 1,562  |  Papers: 230
metric_split
test     1111
cv        383
val        54
train      14


## 7. Colunas derivadas, critérios de aceitação e output final

In [47]:
exp["is_baseline"] = exp["is_baseline_raw"].astype(bool)

# Enforce consistency: is_baseline=True implies balancing_strategy="none"
inconsistent = exp["is_baseline"] & (exp["balancing_strategy"] != "none")
if inconsistent.any():
    exp.loc[inconsistent, "balancing_strategy"] = "none"
    print(f"Fixed {inconsistent.sum()} inconsistent baseline rows")

# within_paper_group_id: groups comparable experiments (same paper, dataset, model)
exp["within_paper_group_id"] = exp.groupby(
    ["paper_id", "dataset_canonical", "model_family"], sort=False
).ngroup()

baseline_groups = set(exp.loc[exp["is_baseline"], "within_paper_group_id"])
exp["has_baseline_in_group"] = exp["within_paper_group_id"].isin(baseline_groups)

# ── Acceptance criteria ────────────────────────────────────────────────────────
PRIMARY_METRICS = ["metric_f1_macro", "metric_f1_weighted", "metric_f1_binary",
                   "metric_balanced_acc", "metric_tpr_gap"]

NUMERIC_DATASET_COLS = ["dataset_size", "dataset_num_classes", "dataset_imbalance_ratio"]
has_2_of_3_numeric = exp[NUMERIC_DATASET_COLS].notna().sum(axis=1) >= 2

mask_accepted = (
    exp[PRIMARY_METRICS].notna().any(axis=1)
    & exp["dataset_canonical"].notna()
    & has_2_of_3_numeric
    & exp["model_family"].notna()
    & exp["balancing_strategy"].notna()
    & (exp["metric_split"] != "train")
)
exp_accepted = exp[mask_accepted].copy().reset_index(drop=True)
exp_accepted["experiment_id"] = exp_accepted.index.map(lambda i: f"exp_{i:05d}")

print(f"Experiments: {len(exp):,} total → {len(exp_accepted):,} accepted ({mask_accepted.mean():.1%})")
print(f"  rejected by 2-of-3 numeric: {(~has_2_of_3_numeric).sum():,}")
print(f"Groups with baseline: {exp_accepted['has_baseline_in_group'].mean():.1%}")

# ── Final schema & save ────────────────────────────────────────────────────────
FINAL_COLS = [
    "experiment_id", "paper_id", "year", "task_type",
    "dataset_canonical", "dataset_size", "dataset_num_classes",
    "dataset_imbalance_ratio", "dataset_is_multilabel",
    "model_family", "model_specific",
    "balancing_strategy", "balancing_strategy_specific", "is_baseline",
    "metric_f1_macro", "metric_f1_weighted", "metric_f1_binary", "metric_f1_micro",
    "metric_balanced_acc", "metric_accuracy", "metric_aucroc", "metric_auprc",
    "metric_gmean", "metric_mcc", "metric_tpr_gap",
    "metric_split", "within_paper_group_id", "has_baseline_in_group",
]

experiments = exp_accepted[FINAL_COLS]
out_path = PROCESSED_DIR / "experiments.parquet"
experiments.to_parquet(out_path, index=False)
print(f"Saved: {out_path}  |  Shape: {experiments.shape}")

Fixed 131 inconsistent baseline rows
Experiments: 1,562 total → 427 accepted (27.3%)
  rejected by 2-of-3 numeric: 396
Groups with baseline: 75.4%
Saved: D:\programas\ufmg\ufmg-2026-1\causal_project\data\processed\experiments.parquet  |  Shape: (427, 28)


In [48]:
raw = pd.read_parquet(INPUT_RAW)
exp_final = pd.read_parquet(PROCESSED_DIR / "experiments.parquet")

n_raw       = len(raw)
n_papers    = raw["paper_id"].nunique()

# metric+split filter
import re
from typing import Optional
def _match_metric(s):
    if pd.isna(s): return None
    s = str(s).lower().strip()
    for col, pat in _METRIC_RULES:
        if re.search(pat, s): return col
    return None
def _match_split(s):
    if pd.isna(s): return None
    s = str(s).lower().strip()
    for lbl, pat in _SPLIT_RULES:
        if re.search(pat, s): return lbl
    return None

has_metric = raw["metric_name_raw"].map(_match_metric).notna()
has_split  = raw["metric_split_raw"].map(_match_split).notna()

n_metric_ok  = has_metric.sum()
n_split_null = raw["metric_split_raw"].isna().sum()
n_split_ok   = has_split.sum()
n_metric_split_ok = (has_metric & has_split).sum()

n_exp_before  = len(exp)          # after pivot
n_exp_final   = len(exp_final)
n_papers_final = exp_final["paper_id"].nunique()

print("=" * 55)
print("  Pipeline: raw → experiments.parquet")
print("=" * 55)
print(f"  Raw configurations (linhas):     {n_raw:>7,}")
print(f"  Papers únicos no raw:            {n_papers:>7,}")
print()
print(f"  Métrica reconhecida:             {n_metric_ok:>7,}  ({n_metric_ok/n_raw:.1%})")
print(f"  Split nulo:                      {n_split_null:>7,}  ({n_split_null/n_raw:.1%})")
print(f"  Split reconhecido:               {n_split_ok:>7,}  ({n_split_ok/n_raw:.1%})")
print(f"  Métrica + split OK:              {n_metric_split_ok:>7,}  ({n_metric_split_ok/n_raw:.1%})")
print()
print(f"  Após pivot (experimentos):       {n_exp_before:>7,}")
print(f"  Após critérios de aceitação:     {n_exp_final:>7,}  ({n_exp_final/n_exp_before:.1%})")
print(f"  Papers representados no final:   {n_papers_final:>7,}  ({n_papers_final/n_papers:.1%})")
print()
print("  Distribuições no dataset final:")
for col in ["task_type", "model_family", "balancing_strategy", "metric_split"]:
    print(f"\n  {col}:")
    for val, cnt in exp_final[col].value_counts().items():
        print(f"    {val:<25} {cnt:>4}  ({cnt/n_exp_final:.1%})")
print()
print(f"  Grupos com baseline:             {exp_final['has_baseline_in_group'].mean():.1%}")
print(f"  Fill rate métricas primárias:")
for col in ["metric_f1_macro", "metric_f1_weighted", "metric_f1_binary", "metric_balanced_acc"]:
    pct = exp_final[col].notna().mean()
    print(f"    {col:<25} {pct:.1%}")

  Pipeline: raw → experiments.parquet
  Raw configurations (linhas):      17,032
  Papers únicos no raw:                512

  Métrica reconhecida:              11,029  (64.8%)
  Split nulo:                        8,739  (51.3%)
  Split reconhecido:                 8,248  (48.4%)
  Métrica + split OK:                4,919  (28.9%)

  Após pivot (experimentos):         1,562
  Após critérios de aceitação:         427  (27.3%)
  Papers representados no final:        71  (13.9%)

  Distribuições no dataset final:

  task_type:
    tabular                    324  (75.9%)
    other                       70  (16.4%)
    vision                      21  (4.9%)
    text                         8  (1.9%)
    time_series                  4  (0.9%)

  model_family:
    ensemble                    89  (20.8%)
    other                       70  (16.4%)
    mlp                         64  (15.0%)
    cnn                         64  (15.0%)
    kernel                      64  (15.0%)
    tree        